# Конспект. Модуль 3: Коллаборативная фильтрация — User-Based (UB-CF)

**Курс:** Мини-курс RecSys (13 модулей)
**Модуль:** 3 из 13 — «Коллаборативная фильтрация: User-Based (UB-CF)»
**Цель модуля:** реализовать первый полноценный рекомендательный алгоритм курса, опираясь на инструментарий Модуля 2 (меры сходства, нормализация) и структуры данных Модуля 1 (user-item матрица). Это первый раз в курсе, где мы переходим от «понимания задачи» к «работающему предсказанию конкретного рейтинга».

**Связь с предыдущими модулями:** формула предсказания в разделе 3.2 напрямую использует корреляцию Пирсона, выведенную и обоснованную в Модуле 2.2.2 — именно поэтому важно было сначала разобраться, почему Пирсон, а не «сырой» косинус, лучше подходит для сравнения пользователей.

## 3.1 Идея, объяснённая на аналогии

### 3.1.1 Базовая аналогия

Представьте, что вы пришли в незнакомый книжный магазин и спрашиваете продавца: «Что бы вы посоветовали?» Хороший продавец сначала уточнит: «А что вам нравилось раньше?» — и, услышав ответ, вспомнит **другого покупателя** с похожими вкусами и посоветует то, что понравилось именно ему.

Это и есть суть **User-Based Collaborative Filtering**: «Если пользователь A и пользователь B совпадали во вкусах в прошлом (оценивали одни и те же товары похожим образом), то товар, который понравился B, но ещё не видел A, скорее всего, понравится и A».

### 3.1.2 Формальная переформулировка

- Это **memory-based** метод (в противовес **model-based** методам вроде матричной факторизации, Модуль 5): модель ничему не обучается заранее в привычном смысле — вместо этого **на каждый запрос** (или через периодически обновляемый кэш) заново ищутся похожие пользователи и на их основе строится предсказание.
- Единственный вход — сама user-item матрица (Модуль 1.4). Никакая информация о содержании товаров не используется (это прямая противоположность контентной фильтрации, Модуль 6).
- Результат работы — либо числовое предсказание рейтинга для конкретной пары (пользователь, товар), либо ранжированный список товаров-кандидатов для рекомендации.

### 3.1.3 Почему это логичное продолжение Модуля 2

В Модуле 2 мы подробно разобрали, **как** измерить, насколько два пользователя похожи (Пирсон, косинус) и **почему** для сравнения именно пользователей предпочтителен Пирсон (устраняет разницу в «щедрости» шкалы). UB-CF — это первое прямое применение этого инструмента: мы буквально берём формулу сходства из Модуля 2.2.2 и используем её как строительный блок алгоритма предсказания.

## 3.2 Алгоритм по шагам

### 3.2.1 Общая схема

1. Для целевого пользователя `u` найти всех пользователей, у которых есть **пересечение** по оценённым товарам с `u` (иначе сходство посчитать невозможно — не с чем сравнивать).
2. Посчитать сходство `sim(u, v)` между `u` и каждым таким пользователем `v` (Пирсон, Модуль 2.2.2).
3. Отобрать `k` ближайших соседей (k-NN) — пользователей с наибольшим сходством.
4. Для товара `i`, который `u` ещё не оценил, сделать взвешенное предсказание на основе оценок соседей.

### 3.2.2 Формула предсказания — разбор по частям

In [ ]:
pred(u, i) = r̄_u + Σ_v∈N(sim(u,v) × (r_vi - r̄_v)) / Σ_v∈N|sim(u,v)|

Разберём каждый элемент формулы отдельно, потому что именно непонимание этой формулы «по кусочкам» — частая причина ошибок при реализации:

- **`r̄_u`** — средний рейтинг целевого пользователя. Это базовая точка отсчёта («сколько бы в среднем поставил этот человек»).
- **`(r_vi - r̄_v)`** — отклонение оценки соседа `v` для товара `i` **от собственного среднего** соседа. Мы не используем сырую оценку `r_vi` напрямую, потому что она искажена «щедростью» соседа — нас интересует не абсолютное значение, а то, **насколько сильнее или слабее обычного** сосед оценил именно этот товар.
- **`sim(u, v)`** — вес, с которым мнение соседа `v` учитывается: чем более похож сосед на целевого пользователя, тем больше его отклонение влияет на итоговое предсказание.
- **Числитель `Σ sim(u,v) × (r_vi - r̄_v)`** — взвешенная сумма отклонений всех соседей, каждое взятое с весом, пропорциональным сходству.
- **Знаменатель `Σ|sim(u,v)|`** — нормировка (важно брать **модуль**, а не саму величину сходства, иначе отрицательные и положительные веса будут «гасить» друг друга в знаменателе, а не отражать реальную суммарную «уверенность» предсказания).
- **Итог:** предсказанный рейтинг = «средняя планка пользователя» + «взвешенная поправка на основе того, насколько сильнее/слабее среднего этот конкретный товар оценили похожие люди».

### 3.2.3 Полный численный пример «от руки»

Возьмём небольшую user-item матрицу — 5 пользователей, 5 фильмов (шкала 1–5):

| | I1 | I2 | I3 | I4 | I5 |
|:---|:---:|:---:|:---:|:---:|:---:|
| **U1 (целевой)** | 5 | 3 | 4 | **?** | — |
| U2 | 4 | 2 | 3 | 4 | — |
| U3 | 1 | 5 | 2 | 1 | 5 |
| U4 | 5 | 3 | 5 | 5 | — |
| U5 | 2 | 2 | 1 | — | 3 |

**Задача:** предсказать, как U1 оценит фильм I4, которого он ещё не видел.

**Шаг 1. Кто вообще оценил I4?** — U2 (4), U3 (1), U4 (5). U5 не оценил I4, поэтому не участвует в предсказании для этого товара (даже если он в целом похож на U1).

**Шаг 2. Считаем Пирсон-сходство U1 с каждым кандидатом**, используя только товары, общие для обоих (I1, I2, I3 — единственные, оценённые и U1, и всеми тремя кандидатами).

*U1 vs U2:*

In [ ]:
U1 = [5, 3, 4], mean = 4  -> центрировано: [1, -1, 0]
U2 = [4, 2, 3], mean = 3  -> центрировано: [1, -1, 0]

sim(U1, U2) = (1×1 + (-1)×(-1) + 0×0) / (√2 × √2) = 2 / 2 = 1.0

Идеальная корреляция — U2 систематически ставит на 1 балл ниже U1 по каждому фильму, то есть их **относительные** предпочтения полностью совпадают.

*U1 vs U3:*

In [ ]:
U1 = [5, 3, 4], mean = 4      -> центрировано: [1, -1, 0]
U3 = [1, 5, 2], mean = 2.667  -> центрировано: [-1.667, 2.333, -0.667]

числитель = 1×(-1.667) + (-1)×2.333 + 0×(-0.667) = -1.667 - 2.333 = -4.0
||U1_c|| = √2 ≈ 1.414
||U3_c|| = √(1.667² + 2.333² + 0.667²) = √8.668 ≈ 2.944

sim(U1, U3) = -4.0 / (1.414 × 2.944) ≈ -0.961

Сильная **отрицательная** корреляция — U3 систематически предпочитает противоположное тому, что нравится U1.

*U1 vs U4:*

In [ ]:
U1 = [5, 3, 4], mean = 4      -> центрировано: [1, -1, 0]
U4 = [5, 3, 5], mean = 4.333  -> центрировано: [0.667, -1.333, 0.667]

числитель = 1×0.667 + (-1)×(-1.333) + 0×0.667 = 0.667 + 1.333 = 2.0
||U4_c|| = √(0.667² + 1.333² + 0.667²) = √2.667 ≈ 1.633

sim(U1, U4) = 2.0 / (1.414 × 1.633) ≈ 0.866

Сильная положительная корреляция, хотя и не идеальная.

**Шаг 3. Отклонения соседей по товару I4 от их собственного среднего:**

In [ ]:
U2: r_{U2,I4} = 4, mean_U2 = 3      -> отклонение = +1.0
U3: r_{U3,I4} = 1, mean_U3 = 2.667  -> отклонение = -1.667
U4: r_{U4,I4} = 5, mean_U4 = 4.333  -> отклонение = +0.667

**Шаг 4. Подставляем всё в формулу предсказания:**

In [ ]:
числитель = sim(U1,U2)×откл_U2 + sim(U1,U3)×откл_U3 + sim(U1,U4)×откл_U4
          = 1.0 × 1.0 + (-0.961) × (-1.667) + 0.866 × 0.667
          = 1.0 + 1.602 + 0.578
          = 3.180

знаменатель = |1.0| + |-0.961| + |0.866| = 2.827

pred(U1, I4) = r̄_U1 + 3.180 / 2.827 = 4 + 1.125 = 5.125

**Важное практическое замечание:** предсказание вышло **за пределы шкалы** (5.125 > 5, максимум по шкале 1–5). Это нормальная ситуация для данной формулы — она нигде явно не ограничивает результат диапазоном исходных значений. На практике всегда добавляют **клиппинг**: `pred = min(max(pred, r_min), r_max)`. В нашем случае: `pred(U1, I4) = 5.0` после клиппинга.

### 3.2.4 Разбор самого интересного момента примера — вклад отрицательно коррелирующего соседа

Обратите особое внимание на вклад U3 в предсказание: `sim(U1,U3) × откл_U3 = (-0.961) × (-1.667) = +1.602` — **положительный** вклад, при том что и сходство, и отклонение по отдельности отрицательны.

**Интерпретация:** U3 имеет вкусы, **противоположные** U1. U3 поставил I4 оценку заметно **ниже** своего обычного уровня (отклонение -1.667 — I4 ему прямо-таки не понравился по его меркам). Раз U3 (с противоположными вкусами) сильно не любит I4, это — **свидетельство в пользу того, что U1 (с противоположными от U3 вкусами) должен любить I4**. Именно поэтому произведение двух отрицательных чисел даёт положительный вклад — и это математически абсолютно корректное, содержательное рассуждение, а не случайность формулы. Это одна из причин, почему при реализации UB-CF **не стоит бездумно отбрасывать соседей с отрицательным сходством** — они несут информацию, просто с противоположным знаком.

На практике, впрочем, некоторые реализации всё же ограничиваются только положительно коррелирующими соседями — это упрощает интерпретацию и снижает риск шума на разреженных данных (когда отрицательная корреляция посчитана на 1-2 общих товарах и статистически ненадёжна — см. 3.3.4). Оба варианта валидны, выбор — вопрос конкретной задачи и объёма данных.

## 3.3 Проблемы User-Based CF

### 3.3.1 Вычислительная сложность — O(N² × M)

Чтобы найти соседей для **каждого** пользователя, в худшем случае нужно сравнить его с **каждым другим** пользователем, а каждое такое сравнение (расчёт Пирсона) стоит `O(M)` (где `M` — число товаров, по которым считается пересечение).

**Конкретный расчёт для реалистичного масштаба** (тот же пример, что в Модуле 1.5.2 и 3.3): 100 000 пользователей, 10 000 товаров.

In [ ]:
Полное попарное сравнение: N² × M = (100 000)² × 10 000 = 10^10 × 10^4 = 10^14 операций

Если процессор выполняет условно `10^9` простых операций в секунду:

In [ ]:
10^14 / 10^9 = 10^5 секунд ≈ 27.8 часов

— **и это только на однократный пересчёт всех сходств**, без учёта того, что данные постоянно обновляются и пересчёт нужно делать регулярно. При 100 миллионах пользователей (реалистичный масштаб крупного маркетплейса) время выросло бы ещё в `1000×` раз — до абсолютно неприемлемых величин.

**Практические способы смягчения (не решающие проблему полностью, но применяемые в реальных системах):**
- **Инвертированный индекс по товарам** (прямая связь с CSC-форматом из Модуля 2.1.4): для каждого товара храним список пользователей, которые его оценили. Тогда для расчёта сходства двух пользователей не нужно сравнивать их со **всеми** остальными — достаточно рассматривать только тех, у кого есть хотя бы одно пересечение по оценённым товарам (что на практике сильно меньше `N`, особенно если пользователь оценил малопопулярные товары).
- **Ограничение выборки соседей** — сравнивать не со всеми пользователями, а только с активными/недавними.
- **Предвычисление и периодическое обновление** (batch job раз в сутки/час) вместо пересчёта в реальном времени.

Несмотря на эти оптимизации, фундаментальная проблема остаётся: UB-CF **структурно** плохо масштабируется на очень большое число пользователей. Это прямая причина, по которой в Модуле 4 мы переходим к Item-Based подходу (число товаров обычно растёт значительно медленнее числа пользователей и они более стабильны), а в Модуле 5 — к матричной факторизации, которая вообще не требует попарных сравнений.

### 3.3.2 Нестабильность вкусов пользователей во времени

Вкусы человека меняются: то, что нравилось год назад, может быть неактуально сегодня. Это означает, что «похожесть» между двумя пользователями, посчитанная на исторических данных, может быть **устаревшей** к моменту, когда мы делаем предсказание.

**Связь с уже известным вам материалом:** это концептуально та же проблема, что Concept Drift и Look-ahead bias, которые вы уже разбирали применительно к временным рядам (Pandas, Недели 1-2 общего плана) — только здесь дрейф происходит не в статистических свойствах данных, а в **предпочтениях людей**. Практическое следствие: сходство между пользователями стоит пересчитывать регулярно и, возможно, с большим весом для более свежих взаимодействий (взвешивание по времени — экспоненциальное затухание значимости старых оценок).

### 3.3.3 Холодный старт пользователя

Прямое применение проблемы из Модуля 1.5.1 к конкретному алгоритму: у нового пользователя нет истории оценок вообще — шаг 1 алгоритма (3.2.1) не может найти **ни одного** пользователя с пересечением по оценённым товарам, потому что пересекаться попросту не с чем. UB-CF в чистом виде **принципиально неприменим** для таких пользователей — необходим fallback (популярность, контентная фильтрация, онбординг-опрос — детали в Модуле 7 и 12.1).

### 3.3.4 Проблема надёжности сходства при малом пересечении

Уже упоминалось в Модуле 2.2.2 как ограничение корреляции Пирсона: если у двух пользователей есть только 1-2 общих оценённых товара, посчитанная на этом сходстве может быть случайно близка к ±1, не отражая реальной устойчивой закономерности.

**Решение — significance weighting (взвешивание по значимости), также называемое shrinkage:**

In [ ]:
sim'(u, v) = sim(u, v) × min(|I_u ∩ I_v|, β) / β

где `|I_u ∩ I_v|` — число товаров, общих для `u` и `v`, а `β` — порог (гиперпараметр, часто выбирают в диапазоне 25–50 в классических работах по CF). Смысл: если общих товаров меньше порога `β`, сходство искусственно занижается пропорционально тому, насколько мало у нас оснований ему доверять; если общих товаров больше `β` — сходство остаётся без изменений.

**Числовой пример:** пусть `sim(u,v) = 0.95` (очень высокое сходство), но посчитано всего на 2 общих товарах, при `β = 50`:

In [ ]:
sim'(u,v) = 0.95 × min(2, 50)/50 = 0.95 × 2/50 = 0.95 × 0.04 = 0.038

После шринкиджа сходство упало почти до нуля — это ровно то поведение, которое мы хотим: «слишком уверенное» сходство, посчитанное на очень малом числе точек, не должно доминировать в предсказании наравне с сходством, подтверждённым десятками общих оценок.

### 3.3.5 Выбор числа соседей k — bias-variance tradeoff

Выбор `k` (число ближайших соседей, шаг 3 алгоритма) — классический компромисс смещения и разброса, с которым вы уже знакомы из логистической регрессии и регуляризации (Неделя 4 общего плана):

- **Малое k** (например, k=5): предсказание основано на очень небольшом числе «голосов» — высокая **дисперсия** (variance): предсказание сильно зависит от случайных особенностей нескольких конкретных соседей, легко переобучиться под шум.
- **Большое k** (например, k=200 или «все подряд»): предсказание усредняется по многим соседям, включая всё менее и менее похожих — высокое **смещение** (bias): предсказание «размывается» к общему среднему, теряя персонализацию.
- **Практика:** оптимальное `k` подбирается кросс-валидацией (тот же принцип, что вы уже применяли для гиперпараметров в sklearn Pipeline, Неделя 5) — типичные хорошие значения на практике часто лежат в диапазоне `k = 20–50`, но это сильно зависит от плотности конкретного датасета.

## 3.4 Практика

### 3.4.1 Полная реализация на Pandas — воспроизведение численного примера из 3.2.3

In [ ]:
import numpy as np
import pandas as pd

# Матрица из раздела 3.2.3
data = {
    'I1': [5, 4, 1, 5, 2],
    'I2': [3, 2, 5, 3, 2],
    'I3': [4, 3, 2, 5, 1],
    'I4': [np.nan, 4, 1, 5, np.nan],
    'I5': [np.nan, np.nan, 5, np.nan, 3],
}
ratings = pd.DataFrame(data, index=['U1', 'U2', 'U3', 'U4', 'U5'])
print(ratings)

def pearson_sim(u_ratings: pd.Series, v_ratings: pd.Series) -> float:
    """Пирсон-сходство по товарам, общим для обоих пользователей (Модуль 2.2.2)."""
    common = u_ratings.notna() & v_ratings.notna()
    if common.sum() < 2:
        return 0.0  # недостаточно данных для расчёта корреляции
    u_common = u_ratings[common]
    v_common = v_ratings[common]
    u_centered = u_common - u_common.mean()
    v_centered = v_common - v_common.mean()
    denom = np.linalg.norm(u_centered) * np.linalg.norm(v_centered)
    if denom == 0:
        return 0.0
    return float(np.dot(u_centered, v_centered) / denom)

def predict_rating(ratings: pd.DataFrame, target_user: str, item: str, k: int = 5) -> float:
    """Полная реализация формулы из раздела 3.2.2."""
    target_ratings = ratings.loc[target_user]
    target_mean = target_ratings.mean()

    # Кандидаты: пользователи, оценившие нужный товар (кроме самого целевого)
    candidates = ratings.index[ratings[item].notna() & (ratings.index != target_user)]

    similarities = {}
    for v in candidates:
        sim = pearson_sim(target_ratings, ratings.loc[v])
        if sim != 0:
            similarities[v] = sim

    # Берём top-k по абсолютному значению сходства (Модуль 3.3.5)
    top_k = sorted(similarities.items(), key=lambda x: -abs(x[1]))[:k]

    if not top_k:
        return target_mean  # fallback: нет подходящих соседей (холодный старт, 3.3.3)

    numerator = sum(
        sim * (ratings.loc[v, item] - ratings.loc[v].mean())
        for v, sim in top_k
    )
    denominator = sum(abs(sim) for _, sim in top_k)

    pred = target_mean + numerator / denominator
    return float(np.clip(pred, 1, 5))  # клиппинг в границы шкалы (3.2.3)

prediction = predict_rating(ratings, 'U1', 'I4', k=5)
print(f"Предсказанный рейтинг U1 для I4: {prediction:.3f}")  # ожидаем 5.0 (после клиппинга 5.125)

### 3.4.2 Significance weighting (shrinkage) — расширение функции сходства

In [ ]:
def pearson_sim_with_shrinkage(u_ratings: pd.Series, v_ratings: pd.Series, beta: int = 25) -> float:
    """Пирсон-сходство с поправкой на число общих товаров (Модуль 3.3.4)."""
    common = u_ratings.notna() & v_ratings.notna()
    n_common = common.sum()
    if n_common < 2:
        return 0.0
    base_sim = pearson_sim(u_ratings, v_ratings)
    shrinkage_factor = min(n_common, beta) / beta
    return base_sim * shrinkage_factor

### 3.4.3 Полномасштабная практика на MovieLens

- Загрузить MovieLens 1M (как в Модуле 1.6), построить полную user-item матрицу для подвыборки из 500-1000 активных пользователей (полная матрица на всех 6000+ пользователей потребует оптимизации из 3.3.1 — начните с подвыборки, чтобы алгоритм отработал за разумное время).
- Реализовать функцию `predict_rating` с векторизацией через матрицу сходства (аналогично `pearson_similarity_matrix` из Модуля 2.4.1), а не через попарные циклы — сравните время выполнения обоих подходов и явно замерьте разницу (`%timeit` в Jupyter).
- **Подбор оптимального k через кросс-валидацию** (прямое применение 3.3.5): для `k ∈ {5, 10, 20, 50, 100}` посчитать RMSE на отложенной выборке (temporal split, как учили в Модуле 1.7 — не случайный!) и построить график `RMSE(k)`. Найти точку, где кривая перестаёт заметно улучшаться (классическая U-образная или монотонно затухающая форма кривой bias-variance).
- **Сравнение с shrinkage и без него:** посчитать RMSE с обычным Пирсоном и с `pearson_sim_with_shrinkage` при разных `β` — показать, что shrinkage особенно помогает на пользователях с малым числом общих оценок.

### 3.4.4 Вопросы для самопроверки

1. Почему в знаменателе формулы предсказания (3.2.2) используется `Σ|sim(u,v)|`, а не просто `Σsim(u,v)`? Что пошло бы не так, если убрать модуль?
2. В примере из 3.2.3 сосед U3 имеет отрицательное сходство с U1, но всё равно положительно повлиял на предсказание. Придумайте свой пример, где отрицательно коррелирующий сосед, наоборot, понизил бы предсказанный рейтинг.
3. Почему UB-CF физически не может дать никакого предсказания, если ни один из пользователей, оценивших целевой товар, не пересекается с целевым пользователем по другим товарам — и что делает в этом случае функция `predict_rating` из раздела 3.4.1 (посмотрите на строку с fallback)?
4. Как связаны между собой проблема из 3.3.1 (сложность O(N²M)) и решение, к которому курс переходит в Модуле 4 — почему Item-Based CF в целом менее подвержен этой проблеме?

## Глоссарий модуля 3

| Термин | Короткое определение |
|:---|:---|
| User-Based CF | Рекомендация на основе похожих пользователей |
| Memory-based метод | Метод без предварительного обучения модели — расчёт «на лету» по сырым данным |
| k-NN (k ближайших соседей) | Отбор k наиболее похожих пользователей для предсказания |
| Отклонение от среднего (deviation) | `r_vi - r̄_v` — насколько сильнее/слабее обычного сосед оценил товар |
| Клиппинг (clipping) | Принудительное ограничение предсказания в допустимый диапазон шкалы |
| Significance weighting / Shrinkage | Занижение сходства, посчитанного на малом числе общих оценок |
| Concept Drift применительно к вкусам | Изменение предпочтений пользователя со временем |
| Bias-Variance Tradeoff (для k) | Компромисс между переобучением на шум (малое k) и излишним усреднением (большое k) |

**Связь со следующим модулем:** Модуль 4 решает главную структурную проблему этого модуля (3.3.1 — вычислительная сложность и нестабильность) за счёт смены объекта сравнения: вместо пользователей сравниваются товары. Формула предсказания в Модуле 4.3 почти зеркальна формуле из 3.2.2, но с ключевым отличием — сходство между **товарами** предвычисляется и кэшируется заранее, а не пересчитывается на каждый запрос.